# PDF Deep Research Pipeline — Walkthrough

This notebook walks through `pdf_main.py` stage by stage so you can see exactly what happens at each step.

```
PDF 파일
  │
  ▼
Stage 0 — Ingestion & Index
  │  pdf_ingestion.py  → PDF → overlapping text chunks
  │  pdf_search.py     → BM25 + OpenAI embeddings → HybridIndex (cached)
  │
  ▼
Stage 1 — Feedback  (선택)
  │  feedback.py       → 1-3 clarifying questions → user answers → combined_query
  │
  ▼
Stage 2 — Deep PDF Research
  │  pdf_research.py   → generate queries → hybrid search → extract learnings → recurse
  │
  ▼
Stage 3 — Reporting
     reporting.py      → synthesise learnings → markdown report → output/output.md
```

The paper used throughout this demo is the **Attention Is All You Need** paper (`1706.03762v7.pdf`).

## Install dependencies

In [ ]:
%pip install -q pypdf rank-bm25 tiktoken openai langchain langchain-openai python-dotenv

## Imports & environment setup

`pdf_search.py` uses the **OpenAI Python SDK** (`client.embeddings.create`) for embeddings.  
`utils.py` uses **LangChain** (`ChatOpenAI`) for text generation — so both are needed.

In [ ]:
import os
import sys

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv()  # reads OPENAI_API_KEY from .env

# Make sure the project root is importable
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils import setup_logging
setup_logging()  # console=INFO, file=DEBUG → output/debug.log

# OpenAI client used by pdf_search.py and pdf_research.py for embeddings + structured output
client = OpenAI()
print("Environment ready.")

## Configuration

Edit this cell to change the PDF, query, or research parameters.

In [ ]:
PDF_PATH        = "1706.03762v7.pdf"        # path to the PDF file
QUERY           = "Transformer 아키텍처와 self-attention 메커니즘이 무엇인지, 어떻게 동작하는지 설명해주세요."
BREADTH         = 2   # number of parallel search queries per recursion level
DEPTH           = 2   # recursion depth  (breadth=2, depth=2 → ~6 LLM calls total)
USE_FEEDBACK    = True
FEEDBACK_MODEL  = "gpt-4o-mini"
RESEARCH_MODEL  = "gpt-4o"
REPORTING_MODEL = "gpt-4o"

---
## Stage 0a — PDF Ingestion

**File:** `step2_pdf_research/pdf_ingestion.py`

`ingest_pdf()` does three things:
1. **`load_pdf_text()`** — extracts raw text page-by-page via `pypdf`. Raises `ValueError` for scanned PDFs (no text layer).
2. **`pick_adaptive_defaults()`** — chooses `chunk_size` and `overlap` based on page count (`<50 pages → 400/60`, `≥50 → 800/100`).
3. **`build_chunks()`** — greedy sentence accumulation:
   - Splits each page into sentences (`split_into_sentences`).
   - Packs sentences into a chunk until the token budget (`chunk_size`) is reached.
   - Copies the last `overlap` tokens from the previous chunk into the next one so context isn't cut off at boundaries.

Each `Chunk` carries: `chunk_id`, `text`, `page_start`, `page_end`, `section_hint`, `token_count`, `source_pdf`.

In [ ]:
from step2_pdf_research.pdf_ingestion import ingest_pdf

chunks, cs, ov = ingest_pdf(PDF_PATH)
# cs = effective chunk_size (tokens), ov = effective overlap (tokens)

print(f"PDF ingested: {PDF_PATH}")
print(f"  Total chunks : {len(chunks)}")
print(f"  chunk_size   : {cs} tokens")
print(f"  overlap      : {ov} tokens")

print("\n--- First 3 chunks ---")
for c in chunks[:3]:
    print(f"\nChunk {c.chunk_id:>3} | pp. {c.page_start}-{c.page_end} | "
          f"section='{c.section_hint}' | {c.token_count} tokens")
    print(f"  {c.text[:200].replace(chr(10), ' ')}...")

---
## Stage 0b — Build Hybrid Index

**File:** `step2_pdf_research/pdf_search.py`

`build_index()` creates a `HybridIndex` with two retrieval methods:

| Method | How | Strength |
|--------|-----|----------|
| **BM25** | TF-IDF-style keyword ranking via `rank_bm25` | Exact term matches |
| **Vector** | `text-embedding-3-small` cosine similarity | Semantic/paraphrase matches |

**Embedding cache:** SHA-256 of `(pdf bytes + chunk_size + overlap)` is used as the cache key.  
Embeddings are saved to `output/pdf_index_<hash>.npz` so re-running is fast (no API calls).

**`search_hybrid()`** fuses both rankings using **Reciprocal Rank Fusion (RRF)**:
```
score(doc) = 0.4 × Σ 1/(k + rank_bm25)  +  0.6 × Σ 1/(k + rank_vec)   (k=60)
```
The higher vector weight (0.6) gives semantic search a slight edge while still catching exact keyword matches.

In [ ]:
from step2_pdf_research.pdf_search import build_index, search_hybrid

# build_index() needs (chunks, client, pdf_path, chunk_size, overlap)
# chunk_size and overlap must match what ingest_pdf returned so the cache key aligns.
index = build_index(chunks, client, PDF_PATH, chunk_size=cs, overlap=ov)

print(f"Index ready")
print(f"  Embeddings shape : {index.embeddings.shape}  (chunks × embed_dim)")
print(f"  Cache file       : {index.cache_path}")

In [ ]:
# Demo: manually run a hybrid search to see what gets retrieved
DEMO_QUERY = "multi-head attention scaled dot-product"
demo_hits = search_hybrid(DEMO_QUERY, index, client, top_k=3)

print(f"Top 3 chunks for: '{DEMO_QUERY}'\n")
for rank, c in enumerate(demo_hits, 1):
    print(f"[{rank}] chunk_id={c.chunk_id} | pp.{c.page_start}-{c.page_end} "
          f"| section='{c.section_hint}' | {c.token_count} tokens")
    print(f"     {c.text[:250].replace(chr(10), ' ')}...\n")

---
## Stage 1 — Feedback (clarifying questions)

**File:** `step1_feedback/feedback.py`

`generate_feedback()` calls `gpt-4o-mini` via LangChain's `with_structured_output()` to generate 1–3 Korean clarifying questions. The answers are concatenated with the original query to form `combined_query`, which is passed to the research stage.

Set `USE_FEEDBACK = False` in the config cell to skip this stage.

In [ ]:
from step1_feedback.feedback import generate_feedback

if USE_FEEDBACK:
    questions = generate_feedback(QUERY, model=FEEDBACK_MODEL, max_feedbacks=3)

    print("Clarifying questions generated:")
    for i, q in enumerate(questions, 1):
        print(f"  {i}. {q}")

    # In the CLI (pdf_main.py) the user types answers interactively.
    # Here we pre-fill answers so the notebook can run non-interactively.
    ANSWERS = [
        "multi-head attention의 수학적 formulation과 직관적 설명에 집중해주세요.",
        "RNN 기반 아키텍처와의 비교도 포함해주세요.",
        "positional encoding이 왜 필요한지 설명해주세요.",
    ]
    answers = ANSWERS[:len(questions)]

    combined_query = f"초기 질문: {QUERY}\n"
    for i, (q, a) in enumerate(zip(questions, answers), 1):
        combined_query += f"\n{i}. 질문: {q}\n   답변: {a}\n"
else:
    combined_query = QUERY

print("\n--- Combined query passed to research stage ---")
print(combined_query)

---
## Stage 2 — Deep PDF Research (recursive)

**File:** `step2_pdf_research/pdf_research.py`

`deep_pdf_research()` is a recursive function. One call = one depth level:

```
deep_pdf_research(query, breadth=2, depth=2)
  │
  ├─ generate_pdf_queries()  → [query_A, query_B]          (breadth=2 queries)
  │
  ├─ For query_A:
  │    search_hybrid(query_A) → [chunk_3, chunk_7, ...]     (top_k=8 chunks)
  │    process_pdf_chunks()   → {learnings: [...], followUpQuestions: [...]}
  │    deep_pdf_research(followUp, breadth=1, depth=1)      (recurse; breadth//2)
  │
  └─ For query_B:
       search_hybrid(query_B) → [chunk_1, chunk_9, ...]     (excludes already-seen ids)
       process_pdf_chunks()   → {learnings: [...], followUpQuestions: [...]}
       deep_pdf_research(followUp, breadth=1, depth=1)      (recurse)
```

**Key design decisions:**
- `seen_chunk_ids` — a `set[int]` threaded through all recursion levels. `search_hybrid()` excludes already-seen chunks so the LLM never re-reads the same text.
- **Adaptive early stop** — if `search_hybrid` returns 0 new chunks for a query, that branch is skipped immediately (no LLM call wasted).
- `process_pdf_chunks()` uses strict grounding: the prompt forbids info outside the provided chunks and requires each learning to end with `(p. N)` page citation.

In [ ]:
from step2_pdf_research.pdf_research import deep_pdf_research

research_results = deep_pdf_research(
    query=combined_query,
    breadth=BREADTH,
    depth=DEPTH,
    client=client,
    model=RESEARCH_MODEL,
    index=index,
)

learnings   = research_results["learnings"]
source_pages = research_results["source_pages"]

print(f"\nResearch complete!")
print(f"  Total unique learnings : {len(learnings)}")
print(f"  Source pages cited     : {len(source_pages)}")

print("\n--- Learnings ---")
for i, l in enumerate(learnings, 1):
    print(f"  {i:>2}. {l}")

print("\n--- Source pages ---")
for s in source_pages:
    print(f"  • {s}")

---
## Stage 3 — Reporting

**File:** `step3_reporting/reporting.py`

`write_final_report()` passes all learnings to `gpt-4o` via LangChain's `llm_call()` (plain text, no structured output) and asks for a detailed Korean markdown report of 6,000+ characters.

The `source_pages` list (which records which PDF pages each chunk came from) is appended as a `## 출처` section at the end.

Note: Unlike the web research pipeline (`main.py`) which passes `visited_urls`, the PDF pipeline passes `source_pages` strings via the same `visited_urls` parameter — the reporting function treats both identically.

In [ ]:
from step3_reporting.reporting import write_final_report

report = write_final_report(
    prompt=combined_query,
    learnings=learnings,
    visited_urls=source_pages,  # source_pages reuses the visited_urls parameter
    model=REPORTING_MODEL,
)

os.makedirs("output", exist_ok=True)
with open("output/output.md", "w", encoding="utf-8") as f:
    f.write(report)

print(f"Report saved to output/output.md ({len(report):,} chars)")

In [ ]:
# Render the report inline
display(Markdown(report))

---
## Done

The full report is at `output/output.md`.

To run the same pipeline as a script (with interactive Korean prompts):
```bash
python pdf_main.py --pdf 1706.03762v7.pdf
```

### Quick parameter guide

| Parameter | Effect | Recommended |
|-----------|--------|-------------|
| `BREADTH` | Parallel queries per recursion level | 2–4 |
| `DEPTH`   | Max recursion levels | 1–3 |
| `BREADTH=4, DEPTH=3` | ~28 LLM calls — thorough but slow | Large documents |
| `BREADTH=2, DEPTH=1` | ~4 LLM calls — fast overview | Quick exploration |